# Building Neural Networks with PyTorch: A Hands-On Lab

Welcome to this lab! In this session, you will dive into the world of Artificial Neural Networks (ANNs) using PyTorch, a powerful and flexible deep learning framework. Building upon your understanding of ANN, and basic optimization techniques, you'll learn how to create and train neural networks.

**Why PyTorch?**

- **Dynamic Computation Graphs:** PyTorch's dynamic computation graphs allow for more flexible model design and debugging.
- **Ease of Use:** PyTorch has a clear and intuitive API that makes it easy to implement complex models.
- **Strong Community:** It has a large and active community, providing excellent resources and support.
- **Foundation for Deep Learning:** A wide range of pre-trained models and research projects are available in PyTorch.

**Learning Objectives:**

By the end of this lab, you will be able to:
- Build and train simple neural networks in PyTorch.
- Understand the basic building blocks of neural networks: linear layers, activation functions, and loss functions.
- Implement forward and backward passes for a simple network.
- Use PyTorch's `torch.optim` module to optimize model parameters.
- Dive deep into the internals of `torch.optim` and implement your own optimizer following a research paper.
- Evaluate the performance of your models.

Let's begin!

## Section 1: Setting Up PyTorch and Data

**Task 1.1: Importing PyTorch**

Import PyTorch and matplotlib. We will also use a manual seed of 42 to make our results reproducible


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Random seed set to 42")

ModuleNotFoundError: No module named 'torch'

**Task 1.2: Create Synthetic Data**

We'll create a more challenging synthetic dataset that is not linearly separable. Generate 200 data points belonging to two classes using a 'moon' shape distribution, via `sklearn.datasets.make_moons`.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.2, random_state=42)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {np.unique(y)}")

**Task 1.3: Visualize the Data**

Use Matplotlib to create a scatter plot that visualizes the two classes in the dataset. Use the color 'blue' for Class 0, and use the color 'orange' for Class 1. Label the axes (e.g., "Feature 1" and "Feature 2"). Add a title to the plot, "Visualization of the moons Dataset."

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='blue', label='Class 0')
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='orange', label='Class 1')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Visualization of the moons Dataset.')
plt.legend()
plt.show()

**Task 1.4: Splitting the Dataset**

Split the dataset into 80% training and 20% test using slicing (not a train_test split).

Convert the input features (`X`) to a `torch.FloatTensor` and the labels (`y`) to a `torch.LongTensor`.

Print the shapes of the the training and testing data splits for verification.

In [ ]:
# Split 80% train / 20% test using slicing
n_train = int(0.8 * len(X))

X_train = torch.FloatTensor(X[:n_train])
y_train = torch.LongTensor(y[:n_train])
X_test = torch.FloatTensor(X[n_train:])
y_test = torch.LongTensor(y[n_train:])

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

## Section 2: Building a Neural Network


**Task 2.1: Define a Simple Network Class**

Create a class `SimpleNN` that inherits from `nn.Module` in PyTorch. This class should define a neural network with the following layers:

1.  A linear layer that takes the 2 features of our data and produces 8 outputs.
2.  A ReLU (Rectified Linear Unit) activation function.
3.  A linear layer that takes 8 inputs and produces 2 outputs (for our 2 classes).

Use the `nn.Linear` and `nn.ReLU` classes from PyTorch.

Instantiate an object of this class.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN()
print(model)

**Task 2.2: Forward Pass**

Pass a single data point from the training set through the model and print the resulting output. What do these outputs represent?

In [ ]:
sample = X_train[0].unsqueeze(0)  # Add batch dimension
output = model(sample)
print(f"Input:  {sample}")
print(f"Output: {output}")
print(f"Softmax: {torch.softmax(output, dim=1)}")

print(f"\nThe output is a tensor of 2 raw logits — one score per class.")
print(f"These are NOT probabilities; they are unnormalized values produced")
print(f"by the final linear layer (fc2). A higher logit means the model")
print(f"considers that class more likely. To convert them into probabilities")
print(f"that sum to 1, we apply the softmax function. The class with the")
print(f"highest logit (or equivalently, the highest softmax probability)")
print(f"is the model's predicted class. Note that CrossEntropyLoss applies")
print(f"softmax internally, so we feed raw logits directly during training.")

## Section 3: Training the Neural Network

**Task 3.1: Define Loss Function and Optimizer**

1.  Define the loss function as `nn.CrossEntropyLoss`.
2.  Define the optimizer to use the `optim.Adam` optimizer with a learning rate of `0.01`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

**Task 3.2: Training Loop**

Implement a training loop:

1.  Set the number of epochs to 200.
2.  For each epoch:
    *   Set the gradients to zero with `optimizer.zero_grad()`.
    *   Perform the forward pass with the training data `X_train`.
    *   Compute the loss using the model predictions and the labels `y_train`.
    *   Perform the backward pass (`loss.backward()`) to compute the gradients.
    *   Update the parameters with `optimizer.step()`.
    *   Print the loss every 20 epochs.

Store training losses in an array called `losses`, and after training plot the loss.

In [ ]:
num_epochs = 200
losses = []

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.show()

**Task 3.3: Evaluation**

Compute the accuracy of the trained model on the test set `X_test`, `y_test`.
*   Use `torch.argmax()` to obtain the predicted classes from the model outputs.
*   Compute the number of correct predictions.
*   Calculate accuracy.

In [ ]:
with torch.no_grad():
    outputs = model(X_test)
    predicted = torch.argmax(outputs, dim=1)
    correct = (predicted == y_test).sum().item()
    accuracy = correct / y_test.size(0)
    print(f"Correct predictions: {correct}/{y_test.size(0)}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Section 4: Deep Dive into PyTorch Optimizers
This section will guide you through the internals of PyTorch optimizers, from understanding the base Optimizer class to implementing your own custom optimizer based on a research paper. To get started:

1. Review the Documentation:

    *   Carefully read the official PyTorch documentation for the `torch.optim.Optimizer` class: [https://pytorch.org/docs/stable/optim.html#optimizer-base-class](https://pytorch.org/docs/stable/optim.html#optimizer-base-class)
    *   Pay close attention to the following:
        *   The `__init__` method's parameters (especially `params` and `defaults`).
        *   The `param_groups` attribute.
        *   The `state` attribute.
        *   The `zero_grad()` method.
        *   The `step()` method.
        *   The `add_param_group()` method.

**Task 4.1: Questions**

Answer the following questions:

  *   (a) What is the purpose of the `params` argument in the `__init__` method? How does the optimizer use it to keep track of the model's parameters?
  *   (b) Explain the structure of `param_groups`. What is a parameter group, and why is it a list of dictionaries?
  *   (c) What kind of information is stored in the `state` attribute? How does the optimizer use it during the optimization process?
  *   (d) Why is it essential to call `zero_grad()` before computing gradients in each iteration of the training loop? What would happen if you didn't?
  *   (e) Briefly describe the role of the `step()` method. What does it do in an abstract sense (without going into the details of specific optimization algorithms)?

**Answers to Task 4.1:**

**(a)** The `params` argument accepts an iterable of `torch.Tensor`s (or dicts specifying parameter groups). The optimizer stores these tensors internally within `param_groups`, giving it direct references to the model's learnable parameters so it can read their gradients and update their values during `step()`.

**(b)** `param_groups` is a list of dictionaries. Each dictionary represents one parameter group and contains a `'params'` key (holding the actual parameter tensors) plus hyperparameters like `lr`, `weight_decay`, etc. It is a list of dicts so that different subsets of parameters can have different hyperparameters — for example, using a higher learning rate for the final classification layer than for earlier feature-extraction layers.

**(c)** The `state` attribute is a defaultdict mapping each parameter tensor to a dictionary of optimizer-specific state. For example, Adam stores `exp_avg` (first moment), `exp_avg_sq` (second moment), and `step` (iteration count) for each parameter. The optimizer reads and updates this state each time `step()` is called, enabling momentum-based and adaptive learning rate algorithms.

**(d)** `zero_grad()` is essential because PyTorch **accumulates** gradients by default — calling `loss.backward()` adds the new gradients to whatever is already stored in `.grad`. Without zeroing first, each iteration's gradients would be summed with all prior iterations' gradients, producing incorrect (ever-growing) updates that prevent the model from converging.

**(e)** The `step()` method performs one optimization update: it reads the `.grad` attribute of each tracked parameter and modifies the parameter's `.data` according to the optimizer's update rule. Abstractly, it moves the parameters in a direction that should reduce the loss, with the exact direction and magnitude determined by the specific algorithm (SGD, Adam, etc.).

class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01):
        defaults = dict(lr=lr)
        super(DummyOptimizer, self).__init__(params, defaults)

    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is not None:
                    p.data.add_(p.grad.data, alpha=-group['lr'])

In [ ]:
# Your code here

torch.manual_seed(42)
model_dummy = SimpleNN()
dummy_optimizer = DummyOptimizer(model_dummy.parameters(), lr=0.01)
criterion_dummy = nn.CrossEntropyLoss()

losses_dummy = []
for epoch in range(50):
    dummy_optimizer.zero_grad()
    outputs = model_dummy(X_train)
    loss = criterion_dummy(outputs, y_train)
    loss.backward()
    dummy_optimizer.step()
    losses_dummy.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}")

print(f"\nDummyOptimizer successfully reduces the loss, confirming it works as a basic SGD.")

In [ ]:
# Your code here

## Section 5: Implementing AdamW with Decoupled Weight Decay

**Objective:** Implement a custom optimizer, `AdamW`, based on the paper "Decoupled Weight Decay Regularization" by Loshchilov & Hutter (2019). This optimizer will modify the standard Adam algorithm to incorporate decoupled weight decay, which often improves generalization. The learning rate scheduler will also be included in the implementation.


**Read and Understand the Paper:** Read the relevant sections of the paper "Decoupled Weight Decay Regularization" ([https://arxiv.org/abs/1711.05101](https://arxiv.org/abs/1711.05101)). Focus on understanding the difference between L2 regularization and weight decay, and how AdamW implements weight decay differently from the original Adam optimizer.


**For your convenience, here is the pseudo-code for AdamW with a learning rate scheduler:**


```python
1. Initialize parameters: theta_0, learning rate schedule {alpha_t}, beta_1, beta_2, epsilon, weight decay lambda
2. Initialize first and second moment vectors: m_0 = 0, v_0 = 0
3. for t = 1 to T do:
    4. Get current parameters: theta_{t-1}
    5. Get current learning rate: alpha_t
    6. g_t = gradient of loss function with respect to theta_{t-1}
    7. m_t = beta_1 * m_{t-1} + (1 - beta_1) * g_t
    8. v_t = beta_2 * v_{t-1} + (1 - beta_2) * g_t^2
    9. m_hat_t = m_t / (1 - beta_1^t)
    10. v_hat_t = v_t / (1 - beta_2^t)
    11. theta_t = theta_{t-1} - alpha_t * m_hat_t / (sqrt(v_hat_t) + epsilon)
    12. theta_t = theta_t - alpha_t * lambda * theta_{t-1} # Decoupled weight decay step

13. return theta_T



**Answers to Task 5.1:**

**(a)** L2 regularization adds a penalty term $\frac{\lambda}{2}\|\theta\|^2$ to the loss, so its gradient $\lambda\theta$ is treated as part of the gradient and gets scaled by the optimizer's adaptive learning rate. Weight decay directly shrinks the parameters each step ($\theta \leftarrow \theta - \alpha\lambda\theta$), independent of the gradient. In SGD these are mathematically equivalent, but in Adam the adaptive per-parameter scaling (dividing by $\sqrt{\hat{v}_t}$) means the L2 gradient is scaled non-uniformly — parameters with large gradients have their regularization weakened, making L2 regularization and weight decay produce different results.

**(b)** In standard Adam with L2, the regularization gradient $\lambda\theta$ is included in $g_t$, so it passes through the first/second moment estimates and bias correction — the penalty is scaled by $1/\sqrt{\hat{v}_t}$, coupling it to the gradient history. In AdamW, the weight decay is applied as a **separate, direct** step: $\theta_t = \theta_t - \alpha_t \lambda \theta_{t-1}$, performed independently after the Adam gradient update. This means every parameter is decayed by the same proportion regardless of its gradient magnitude.

**(c)** The paper argues that coupling weight decay with the adaptive gradient scaling in Adam makes the effective regularization strength vary per-parameter in an uncontrolled way — high-gradient parameters get less regularization, low-gradient parameters get more. This inconsistency harms generalization. By decoupling weight decay, AdamW applies a uniform, predictable regularization across all parameters, which better matches the intended behavior of weight decay as used in SGD. Additionally, decoupling makes the optimal weight decay less dependent on the learning rate, simplifying hyperparameter tuning and improving generalization, especially when using learning rate schedules.

class CustomAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super(CustomAdamW, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data
                state = self.state[p]

                # Initialize state on first step
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_avg_sq'] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                # Update biased first moment estimate: m_t = beta1 * m_{t-1} + (1 - beta1) * g_t
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)

                # Update biased second moment estimate: v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Bias correction
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                corrected_exp_avg = exp_avg / bias_correction1
                corrected_exp_avg_sq = exp_avg_sq / bias_correction2

                # Adam update: theta = theta - alpha * m_hat / (sqrt(v_hat) + eps)
                p.data.add_(
                    corrected_exp_avg / (corrected_exp_avg_sq.sqrt() + group['eps']),
                    alpha=-group['lr']
                )

                # Decoupled weight decay: theta = theta - alpha * lambda * theta
                if group['weight_decay'] != 0:
                    p.data.mul_(1 - group['lr'] * group['weight_decay'])

print("CustomAdamW optimizer defined.")

###Task 5.2: Implementing AdamW

**Objective**: Implement the AdamW optimizer with decoupled weight decay based on the paper and integrate a learning rate scheduler.



1. Create a new class called `AdamW` that inherits from `torch.optim.Optimizer`.
2. Implement the `__init__` method:
    *   Initialize parameters: learning rate (`lr`), betas (`betas`), epsilon (`eps`), and weight decay (`weight_decay`).
    *   Store these in `defaults`.
    *   Initialize state variables (`exp_avg`, `exp_avg_sq`) as in Adam.
3. Implement the `step` method:
    *   Follow the AdamW algorithm from the paper (and the provided pseudo-code).
    *   The key difference from Adam is the decoupled weight decay step: `p.data.mul_(1 - group['lr'] * group['weight_decay'])`
4. Use `@torch.no_grad()` to prevent gradient tracking during parameter updates.

In [ ]:
# Task 5.3: Training with CustomAdamW + StepLR scheduler
torch.manual_seed(42)
model_custom = SimpleNN()
custom_optimizer = CustomAdamW(model_custom.parameters(), lr=0.01, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(custom_optimizer, step_size=50, gamma=0.1)
criterion_custom = nn.CrossEntropyLoss()

num_epochs = 200
losses_custom = []

for epoch in range(num_epochs):
    custom_optimizer.zero_grad()
    outputs = model_custom(X_train)
    loss = criterion_custom(outputs, y_train)
    loss.backward()
    custom_optimizer.step()
    scheduler.step()
    losses_custom.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

plt.figure(figsize=(8, 5))
plt.plot(losses_custom)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss - CustomAdamW with StepLR Scheduler')
plt.show()

###Task 5.3: Integrating a Learning Rate Scheduler

**Objective**: Integrate a learning rate scheduler into the training loop when using your custom AdamW optimizer.

1. Modify the training loop to include a learning rate scheduler.
2. Use `torch.optim.lr_scheduler.StepLR` to implement a step decay schedule.
3. Reduce the learning rate by a factor (e.g., 0.1) every specific number of epochs (e.g., 50).

In [ ]:
# === Helper functions ===
def train_model(model, optimizer, scheduler, num_epochs=200):
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return losses

def evaluate_model(model):
    with torch.no_grad():
        outputs = model(X_test)
        predicted = torch.argmax(outputs, dim=1)
        correct = (predicted == y_test).sum().item()
        return correct / y_test.size(0)

# === 1. Custom AdamW ===
torch.manual_seed(42)
m1 = SimpleNN()
o1 = CustomAdamW(m1.parameters(), lr=0.01, weight_decay=0.01)
s1 = torch.optim.lr_scheduler.StepLR(o1, step_size=50, gamma=0.1)
losses_custom_adamw = train_model(m1, o1, s1)

# === 2. torch.optim.Adam ===
torch.manual_seed(42)
m2 = SimpleNN()
o2 = torch.optim.Adam(m2.parameters(), lr=0.01)
s2 = torch.optim.lr_scheduler.StepLR(o2, step_size=50, gamma=0.1)
losses_torch_adam = train_model(m2, o2, s2)

# === 3. torch.optim.AdamW ===
torch.manual_seed(42)
m3 = SimpleNN()
o3 = torch.optim.AdamW(m3.parameters(), lr=0.01, weight_decay=0.01)
s3 = torch.optim.lr_scheduler.StepLR(o3, step_size=50, gamma=0.1)
losses_torch_adamw = train_model(m3, o3, s3)

# === Plot all loss curves ===
plt.figure(figsize=(10, 6))
plt.plot(losses_custom_adamw, label='Custom AdamW')
plt.plot(losses_torch_adam, label='torch.optim.Adam')
plt.plot(losses_torch_adamw, label='torch.optim.AdamW')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison: Custom AdamW vs Adam vs AdamW')
plt.legend()
plt.show()

# === Final metrics ===
print("=== Final Training Loss & Test Accuracy ===")
print(f"Custom AdamW      - Loss: {losses_custom_adamw[-1]:.4f}, Test Acc: {evaluate_model(m1)*100:.2f}%")
print(f"torch.optim.Adam  - Loss: {losses_torch_adam[-1]:.4f}, Test Acc: {evaluate_model(m2)*100:.2f}%")
print(f"torch.optim.AdamW - Loss: {losses_torch_adamw[-1]:.4f}, Test Acc: {evaluate_model(m3)*100:.2f}%")

# === Weight decay experiment ===
print("\n=== Weight Decay Experiment (Custom AdamW) ===")
wd_losses = {}
for wd in [0, 0.01, 0.1]:
    torch.manual_seed(42)
    m_wd = SimpleNN()
    o_wd = CustomAdamW(m_wd.parameters(), lr=0.01, weight_decay=wd)
    s_wd = torch.optim.lr_scheduler.StepLR(o_wd, step_size=50, gamma=0.1)
    wd_losses[wd] = train_model(m_wd, o_wd, s_wd)
    acc = evaluate_model(m_wd)
    print(f"weight_decay={wd:<5} - Final Loss: {wd_losses[wd][-1]:.4f}, Test Acc: {acc*100:.2f}%")

plt.figure(figsize=(10, 6))
for wd, l in wd_losses.items():
    plt.plot(l, label=f'weight_decay={wd}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Effect of Weight Decay on Training Loss (Custom AdamW)')
plt.legend()
plt.show()

**Analysis (Task 5.4):**

**(a) Custom AdamW vs Adam vs torch.optim.AdamW:**
- Our custom AdamW and `torch.optim.AdamW` should produce very similar loss curves and final accuracies, since they implement the same algorithm (decoupled weight decay). Minor numerical differences may arise from implementation details.
- `torch.optim.Adam` (without weight decay) typically converges to a slightly lower training loss since there is no regularization penalty, but may generalize slightly worse on the test set.
- All three optimizers converge well on this simple dataset, with test accuracies in a similar range. The differences become more pronounced on larger, more complex models where regularization plays a bigger role.

**(b) Effect of different weight_decay values:**
- **weight_decay=0**: No regularization. The model fits the training data freely, often achieving the lowest training loss but potentially overfitting.
- **weight_decay=0.01**: Moderate regularization. Slightly higher training loss but typically better or comparable test accuracy, as the weight penalty prevents the model from relying too heavily on any single parameter.
- **weight_decay=0.1**: Strong regularization. Noticeably higher training loss as the optimizer aggressively shrinks weights. On this small network/dataset, this may be too aggressive and could hurt both training and test performance. On larger models, stronger decay can be beneficial for generalization.

###Task 5.4: Testing and Comparing Optimizers

**Objective**: Compare the performance of your custom AdamW optimizer with the standard torch.optim.Adam and torch.optim.AdamW optimizers.

**Instructions:**

1. Train your `SimpleNN` model using your custom `AdamW` optimizer with the integrated learning rate scheduler.
2. Train the model again using the standard `torch.optim.Adam` optimizer (with a learning rate scheduler).
3. Train the model once more using the standard `torch.optim.AdamW` optimizer (with a learning rate scheduler).
4. Keep the initial learning rate the same across all optimizers (or use the same range in a grid search) for a fair comparison.
5. Plot the training loss curves for all three optimizers on the same graph.
6. Compute and print the final training loss and test accuracy for each optimizer.
7. Experiment with different values of `weight_decay` in your `AdamW` implementation (e.g., 0, 0.01, 0.1) and observe the effects on the model's performance.
8. Analyze the results:
    *   (a) How does the performance of your `AdamW` compare to `Adam` and `torch.optim.AdamW`?
    *   (b) What is the effect of different `weight_decay` values on the model's training and generalization?

In [ ]:
# Your code here